# Entraînement de l'agent Rocket League sur Google Colab

**Avant la 1re session** (voir `cloud/README.md`) :
1. Sur le portable : `python scripts/make_cloud_bundle.py --init runs/s5_spo_laptop`.
2. Dans ton Google Drive, dossier `rl-agent/` : `rl-agent-code.zip`, `rl-agent-init.zip` et ton dossier `collision_meshes/`.
3. *Exécution → Modifier le type d'exécution* → **GPU T4**. Puis *Exécuter tout*.

Les checkpoints sont écrits dans `Drive/rl-agent/runs/` : si la session coupe, relancer le notebook reprend le run.
⚠️ Colab gratuit : garder l'onglet ouvert (déconnexion si inactif), 2 vCPU seulement (collecte lente).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/rl-agent"
RUN_NAME = "s7b_spo_colab"
PROFILES = "s7 s7b"                          # réglages d'apprentissage (configs/*.yaml)
HOURS = 11
AUTOTUNE = True                              # mesures en début de session → config la plus rapide
RUNS_DIR = f"{DRIVE}/runs"

In [ ]:
import glob, os, shutil, subprocess, zipfile

def find_dir_with(root, marker):
    """Dossier sous root qui contient `marker` (gère les zips décompressés ou non)."""
    for z in glob.glob(os.path.join(root, "**", "*.zip"), recursive=True):
        dest = os.path.join("/tmp/unzipped", os.path.basename(z)[:-4])
        if not os.path.exists(dest):
            zipfile.ZipFile(z).extractall(dest)
    for base in (root, "/tmp/unzipped"):
        hits = sorted(glob.glob(os.path.join(base, "**", marker), recursive=True), key=len)
        if hits:
            return os.path.dirname(hits[0])
    return None


def where(setting, marker, root="/kaggle/input"):
    """setting = "auto" (cherche partout sous /kaggle/input) ou un chemin explicite."""
    return find_dir_with(root if setting == "auto" else setting, marker)

code_dir = find_dir_with(DRIVE, "pyproject.toml")
init_dir = find_dir_with(DRIVE, "ckpt.pt")
assert code_dir and os.path.isdir(f"{DRIVE}/collision_meshes"), "il manque le code ou collision_meshes/ dans Drive/rl-agent"
shutil.copytree(code_dir, "/content/rl-agent", dirs_exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
print("code :", code_dir, "| init :", init_dir)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
env = dict(os.environ, CODE_DIR="/content/rl-agent", MESHES_DIR=f"{DRIVE}/collision_meshes", RUNS_DIR=RUNS_DIR,
           PLATFORM="colab", RUN_NAME=RUN_NAME, PROFILES=PROFILES, HOURS=str(HOURS),
           AUTOTUNE="1" if AUTOTUNE else "0", INIT_DIR=init_dir or "", VENV_DIR="/content/rlenv")
script = "/content/rl-agent/cloud/run_cloud.sh"
data = open(script, "rb").read()                     # lire AVANT d'ouvrir en écriture
open(script, "wb").write(data.replace(bytes([13, 10]), bytes([10])))   # fins de ligne Linux
p = subprocess.Popen(["bash", script], env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
try:
    for line in p.stdout:
        if not line.startswith("it ") or line.split("|")[0].strip().endswith("0"):
            print(line, end="", flush=True)    # toutes les 10 itérations + messages
    p.wait()
except KeyboardInterrupt:                      # bouton « Stop » : arrêt propre avec checkpoint
    open(f"{RUNS_DIR}/{RUN_NAME}/STOP", "w").close()
    p.wait()
print("code de sortie :", p.returncode)

In [ ]:
import json
rows = [json.loads(l) for l in open(f"{RUNS_DIR}/{RUN_NAME}/metrics.jsonl")]
last = rows[-1]
print(f"steps {last['global_step']:,.0f} | elo {last['elo']:.0f} | entropie {last['learn/entropy']:.3f}")
for k in sorted(k for k in last if k.startswith(("drill/", "mech/")) and not k.endswith("episodes")):
    print(f"{k:35s} {last[k]:.3f}")